In [ ]:
# Cribl Python SDK example notebook
#
# This notebook shows:
# 1) how to call common Cribl APIs with `cribl-control-plane`
# 2) how to prepare discovered API objects for plotting
# 3) how to use AI + Jinja prompt templates for visualization generation

import micropip

await micropip.install("cribl-control-plane")

In [ ]:
import os

server_url = os.getenv("CRIBL_API_URL")
if not server_url:
    raise RuntimeError("CRIBL_API_URL is not set in this runtime.")

print(f"CRIBL_API_URL={server_url}")
server_url

In [ ]:
# Synchronous SDK example: fetch common objects you can inspect and visualize.
from cribl_control_plane import CriblControlPlane

health = {}
sources = None
destinations = None
pipelines = None
api_summary = {}

with CriblControlPlane(server_url=server_url) as ccp_client:
    # Health endpoint
    health = ccp_client.health.get()

    # Most Cribl config APIs are group-scoped
    worker_group_id = "default"
    group_url = f"{server_url}/m/{worker_group_id}"

    # Inventory endpoints
    sources = ccp_client.sources.list(server_url=group_url)
    destinations = ccp_client.destinations.list(server_url=group_url)
    pipelines = ccp_client.pipelines.list(server_url=group_url)

api_summary = {
    "group": "default",
    "health_status": getattr(getattr(health, "status", None), "value", str(getattr(health, "status", "unknown"))),
    "source_count": len((sources.items if sources else None) or []),
    "destination_count": len((destinations.items if destinations else None) or []),
    "pipeline_count": len((pipelines.items if pipelines else None) or []),
}

print(f"Server health: {health}")
print("Inventory summary:", api_summary)
api_summary

In [ ]:
# ### Prompt:
# Generate Python code that visualizes Cribl API objects with matplotlib.
# Use this server health object for context: {{ health | describe }}
# Use this inventory summary dict for plotting: {{ api_summary | describe }}
# Build a bar chart for source_count, destination_count, and pipeline_count
# and print a short textual interpretation.


### Manual visualization example

This cell demonstrates a direct matplotlib chart from API data. It is useful as a baseline before using AI-generated code.

In [ ]:
import matplotlib.pyplot as plt

labels = ["sources", "destinations", "pipelines"]
values = [
    api_summary["source_count"],
    api_summary["destination_count"],
    api_summary["pipeline_count"],
]

plt.figure(figsize=(7, 4))
plt.bar(labels, values)
plt.title(f"Cribl object counts ({api_summary['group']})")
plt.ylabel("count")
plt.tight_layout()
plt.show()

values

### AI + Jinja prompt patterns

Use the AI button on this cell and keep prompts focused on variables already present in kernel memory.

- `{{ health }}` or `{{ health | describe }}` for status detail
- `{{ api_summary }}` or `{{ api_summary | describe }}` for compact numeric context

`| describe` is usually better for large/typed objects because it gives structured summaries the model can reason about.

In [ ]:
# ### Prompt:
# Generate Python code that discovers what to visualize from:
#   - health: {{ health | describe }}
#   - api_summary: {{ api_summary | describe }}
#
# Requirements:
# - choose an appropriate matplotlib chart
# - include axis labels and title
# - print 2-3 lines explaining what the plot indicates
# - keep code self-contained and executable in this notebook